<div align="center">

# NYC Citi Bike Demand Forecasting

### *Forecasting Daily Ride Volume with Time-Series Models*

**CUSP-GX 7023 · Applied Data Science · Spring 2026**

**Group: Apex** &nbsp;|&nbsp; Jerry Zhao · Clement Mo · Keyang Yan

</div>

---

# 1. Question & Hypothesis

## 1.1 Background

NYC Citi Bike is the largest public bike-sharing system in the United States. Each trip is recorded with timestamps, station identifiers, and rider categories, making it a strong source for studying **urban mobility** through time-series analysis. Reliable forecasts of daily bike demand have direct value for **fleet rebalancing**, **station-level planning**, and **low-emission transport policy**.

The full-year raw dataset contains approximately **45.8 million trip records**, which may exceed the memory limits of a standard laptop if loaded all at once. We therefore use a **chunk-based aggregation pipeline**. For each monthly Citi Bike file, the notebook downloads the official ZIP to a local cache if needed, reads only the columns needed for this project, mainly `started_at` and `member_casual`, processes each CSV in chunks, aggregates rides into daily and hourly counts by user type, and discards raw trip-level rows after aggregation.

The final analysis is based on two compact tables: `daily_usage` and `hourly_usage`, with separate columns for member, casual, and total rides. This greatly reduces memory usage while preserving the information needed for city-wide daily/hourly usage analysis and forecasting.

## 1.2 Main Research Question

> **Can historical Citi Bike trip data be used to forecast future daily ride volume in New York City?**

## 1.3 Hypotheses

| # | Hypothesis |
|:-:|:--|
| **H1** | The series decomposes into a slowly varying **trend**, a **weekly seasonal** component (period = 7), and a residual. |
| **H2** | Weekday usage shows pronounced **morning and evening commute peaks**; weekend usage is flatter and shifts toward midday. |
| **H3** | **Member** rides concentrate on weekday commuting hours; **casual** rides concentrate on weekend afternoons. |
| **H4** | A simple **ARIMA(p, d, q)** model, informed by ACF / PACF diagnostics, outperforms a naive last-day forecast on MAE, MSE, and RMSE. |
| **H5** | A **linear regression** of daily ride counts on calendar features (day-of-week, month, `is_weekend`) and **lag features** (lag-1, lag-7, 7-day rolling mean) also outperforms the naive baseline. |

## 1.4 Scope and Limitations

- **Temporal scope**: Jan 1 - Dec 31, 2025 (NYC only; `JC*` Jersey City files excluded).
- **No exogenous variables**: weather and public-holiday effects are intentionally omitted; the forecasting setup is univariate.
- **Aggregation level**: city-wide daily and hourly counts. Station-level spatial analysis is out of scope.
- **Forecast horizon**: short-term monthly holdout tests, using June 2025 and December 2025 as two separate test months.

## 1.5 Success Criteria

On the held-out test months (**June 2025** and **December 2025**), a successful model should:

1. Achieve **lower MAE, MSE, and RMSE** than the naive last-day baseline.
2. Have residuals that show no strong systematic pattern, with most residuals staying within a **+/-3 sigma band** and any exceptions explicitly discussed.

Comparing the ARIMA model (**H4**) and the regression model (**H5**), and explaining their respective strengths, is itself part of the analysis.

---

# 2. Data

## Step 0 · Setup & Imports

Standard imports for data handling, visualization, time-series modeling, and regression. Sets a fixed random seed and a project-level `data/` directory.

In [ ]:
# ------------------------
# 1. Data handling and visualization
# ------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------
# 2. Streaming download (Python standard library)
# ------------------------
import io
import zipfile
from urllib.request import urlopen
from pathlib import Path

# ------------------------
# 3. Modeling libraries
# ------------------------
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------
# 4. Display and plotting settings
# ------------------------
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 4)

np.random.seed(42)

# ------------------------
# 5. Project paths
# ------------------------
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print('Setup complete.')


## Step 1 · Load & Aggregate Data

Each monthly ZIP is retrieved from the official Citi Bike S3 bucket and cached locally if it is not already present in `data/`. The notebook then opens every NYC CSV inside the ZIP, skips `JC*` Jersey City files, reads only `started_at` and `member_casual` in chunks, aggregates to daily and hourly ride counts by user type, and discards raw rows immediately after each chunk.

Outputs saved to `data/`: `daily_usage.csv` and `hourly_usage.csv`.


In [ ]:
# ------------------------
# 1. Use cached aggregated files when available
# ------------------------
daily_path = DATA_DIR / "daily_usage.csv"
hourly_path = DATA_DIR / "hourly_usage.csv"

if daily_path.exists() and hourly_path.exists():
    print("Aggregated CSVs already exist; skipping raw trip download and aggregation.")
    print("Daily file:", daily_path)
    print("Hourly file:", hourly_path)

else:
    # ------------------------
    # 2. Define months and download URL
    # ------------------------
    months = [f"2025{m:02d}" for m in range(1, 13)]
    URL = "https://s3.amazonaws.com/tripdata/{ym}-citibike-tripdata.zip"

    daily_parts = []
    hourly_parts = []
    total_trips = 0

    # ------------------------
    # 3. Stream each monthly ZIP and aggregate in chunks
    # ------------------------
    for ym in months:
        zip_path = DATA_DIR / f"{ym}-citibike-tripdata.zip"
        print(f"Processing {ym} ...")

        if zip_path.exists():
            zip_source = zip_path
        else:
            print(f"  Downloading {ym} ...")
            with urlopen(URL.format(ym=ym), timeout=120) as response:
                zip_source = io.BytesIO(response.read())

        with zipfile.ZipFile(zip_source) as zf:
            csv_names = [
                name
                for name in zf.namelist()
                if name.endswith(".csv")
                and Path(name).name.startswith(ym)
                and not Path(name).name.startswith("JC")
            ]

            for csv_name in csv_names:
                print(" ", Path(csv_name).name)
                with zf.open(csv_name) as f:
                    for chunk in pd.read_csv(
                        f,
                        usecols=["started_at", "member_casual"],
                        parse_dates=["started_at"],
                        chunksize=1_000_000,
                    ):
                        chunk = chunk.dropna(subset=["started_at", "member_casual"])
                        chunk["date"] = chunk["started_at"].dt.date
                        chunk["hour"] = chunk["started_at"].dt.hour
                        total_trips += len(chunk)

                        daily_parts.append(
                            chunk.groupby(["date", "member_casual"]).size().rename("count")
                        )
                        hourly_parts.append(
                            chunk.groupby(["date", "hour", "member_casual"]).size().rename("count")
                        )

    # ------------------------
    # 4. Combine chunk-level aggregations and save compact outputs
    # ------------------------
    daily_usage = (
        pd.concat(daily_parts)
        .groupby(level=["date", "member_casual"])
        .sum()
        .unstack(fill_value=0)
    )
    hourly_usage = (
        pd.concat(hourly_parts)
        .groupby(level=["date", "hour", "member_casual"])
        .sum()
        .unstack(fill_value=0)
    )

    for usage in [daily_usage, hourly_usage]:
        for col in ["casual", "member"]:
            if col not in usage.columns:
                usage[col] = 0
        usage["total"] = usage["casual"] + usage["member"]

    daily_usage = daily_usage[["casual", "member", "total"]]
    hourly_usage = hourly_usage[["casual", "member", "total"]]

    daily_usage.index = pd.to_datetime(daily_usage.index)
    daily_usage.index.name = "date"

    daily_usage.to_csv(daily_path)
    hourly_usage.to_csv(hourly_path)

    print("Total trips processed:", total_trips)
    print("daily_usage shape:", daily_usage.shape)
    print("hourly_usage shape:", hourly_usage.shape)
    print("date range:", daily_usage.index.min().date(), daily_usage.index.max().date())

# 3. Descriptive Analysis

## Step 2 · Data Cleaning & Validation

Load the aggregated CSVs from disk, filter to Jan 1 – Dec 31 2025, and check for missing dates, zero-ride days, and outliers.

In [ ]:
# ------------------------
# 1. Load CSVs from disk
# ------------------------
daily  = pd.read_csv(DATA_DIR / 'daily_usage.csv',  index_col=0, parse_dates=True)
hourly = pd.read_csv(DATA_DIR / 'hourly_usage.csv', index_col=[0, 1], parse_dates=['date'])

rows_before = len(daily)

# ------------------------
# 2. Filter to 2025-01-01 ~ 2025-12-31
# ------------------------
daily  = daily.loc['2025-01-01':'2025-12-31']

hourly_dates = hourly.index.get_level_values('date')
hourly = hourly[(hourly_dates >= '2025-01-01') & (hourly_dates <= '2025-12-31')]

rows_after = len(daily)

# ------------------------
# 3. Check for missing dates
# ------------------------
full_range    = pd.date_range('2025-01-01', '2025-12-31', freq='D')
missing_dates = full_range.difference(daily.index)

# ------------------------
# 4. Check for zero-ride days
# ------------------------
zero_days = daily[daily['total'] == 0]

# ------------------------
# 5. Check for outliers (beyond ±3 standard deviations)
# ------------------------
mean_total   = daily['total'].mean()
std_total    = daily['total'].std()
upper_bound  = mean_total + 3 * std_total
lower_bound  = mean_total - 3 * std_total
outlier_days = daily[(daily['total'] > upper_bound) | (daily['total'] < lower_bound)]

# ------------------------
# 6. Print summary
# ------------------------
print('Rows before filter :', rows_before)
print('Rows after filter  :', rows_after)
print('Missing dates      :', len(missing_dates))
if len(missing_dates) > 0:
    print('  Dates:', list(missing_dates.date))
print('Zero-ride days     :', len(zero_days))
print('Outlier days (±3σ) :', len(outlier_days))
if len(outlier_days) > 0:
    print(outlier_days[['total']])

print('\nCleaning complete.')


## Step 3 · Basic Statistics

Check the shape, date range, data types, and summary statistics of the daily dataset.

In [ ]:
# ------------------------
# 1. Shape and date range
# ------------------------
print('Shape:', daily.shape)
print('Date range:', daily.index.min().date(), 'to', daily.index.max().date())
print()

# ------------------------
# 2. Data types
# ------------------------
print('Data types:')
print(daily.dtypes)
print()

# ------------------------
# 3. Descriptive statistics
# ------------------------
print('Descriptive statistics:')
print(daily.describe().round(0))


## Step 4 · Member vs Casual Breakdown

Compare the total number of rides by user type over the full Jan–Dec 2025 period.

In [ ]:
# ------------------------
# 1. Calculate totals
# ------------------------
total_member = daily['member'].sum()
total_casual = daily['casual'].sum()

print('Total member rides:', total_member)
print('Total casual rides:', total_casual)

# ------------------------
# 2. Bar chart
# ------------------------
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Member', 'Casual'], [total_member, total_casual], color=['steelblue', 'coral'])
ax.set_title('Total Rides by User Type (Jan–Dec 2025)')
ax.set_ylabel('Total Rides')
ax.set_xlabel('User Type')

# Add value labels on top of bars
ax.text(0, total_member + 50000, f'{total_member:,}', ha='center', fontsize=10)
ax.text(1, total_casual + 50000, f'{total_casual:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()


## Step 5 · Daily Ride Count Over Time

Line plot of total daily rides and a monthly bar chart to show the seasonal growth trend.

In [ ]:
# ------------------------
# 1. Daily total rides line plot
# ------------------------
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily["total"], color="steelblue", linewidth=1)
ax.set_title("Daily Citi Bike Rides — NYC (Jan–Dec 2025)")
ax.set_xlabel("Date")
ax.set_ylabel("Total Rides")
plt.tight_layout()
plt.show()

# ------------------------
# 2. Monthly total rides bar chart
# ------------------------
monthly = daily["total"].resample("ME").sum()
month_labels = monthly.index.strftime("%b")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(month_labels, monthly.values, color="steelblue")
ax.set_title("Monthly Total Rides — NYC (Jan–Dec 2025)")
ax.set_xlabel("Month")
ax.set_ylabel("Total Rides")
for i, v in enumerate(monthly.values):
    ax.text(i, v + 10000, f"{v:,.0f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


## Step 5B · Trend, Weekly Seasonality & Residuals

Decompose the daily ride series into a 7-day rolling trend, an average weekday seasonal component, and a residual. This directly checks H1 from the research hypotheses.


In [ ]:
# ------------------------
# 1. Estimate trend with a centered 7-day rolling average
# ------------------------
decomp = daily[["total"]].copy()
decomp["trend_7d"] = decomp["total"].rolling(window=7, center=True).mean()

# ------------------------
# 2. Estimate weekly seasonal component
# ------------------------
decomp["weekday"] = decomp.index.dayofweek
decomp["detrended"] = decomp["total"] - decomp["trend_7d"]
weekly_seasonal = decomp.dropna().groupby("weekday")["detrended"].mean()
decomp["seasonal_weekly"] = decomp["weekday"].map(weekly_seasonal)

# ------------------------
# 3. Calculate residuals
# ------------------------
decomp["residual"] = decomp["total"] - decomp["trend_7d"] - decomp["seasonal_weekly"]

print("Weekly seasonal component:")
print(weekly_seasonal.rename(index={0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}).round(0))
print()
print("Residual mean:", round(decomp["residual"].mean(), 1))
print("Residual std :", round(decomp["residual"].std(), 1))

# ------------------------
# 4. Plot observed series, trend, seasonality, and residuals
# ------------------------
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=False)

axes[0].plot(decomp.index, decomp["total"], label="Observed", color="lightgray")
axes[0].plot(decomp.index, decomp["trend_7d"], label="7-day Trend", color="steelblue", linewidth=2)
axes[0].set_title("Daily Rides and 7-Day Rolling Trend")
axes[0].set_ylabel("Total Rides")
axes[0].legend()

axes[1].bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], weekly_seasonal.values, color="coral")
axes[1].set_title("Estimated Weekly Seasonal Component")
axes[1].set_ylabel("Deviation from Trend")

axes[2].plot(decomp.index, decomp["residual"], color="darkorange")
axes[2].axhline(0, color="black", linewidth=1)
axes[2].set_title("Residuals After Removing Trend and Weekly Seasonality")
axes[2].set_xlabel("Date")
axes[2].set_ylabel("Residual")

plt.tight_layout()
plt.show()


# 4. Exploratory Analysis

## Step 6 · Weekday vs Weekend Usage

Compare average daily rides on weekdays versus weekends.

In [ ]:
# ------------------------
# 1. Label each day as weekday or weekend
# ------------------------
daily['weekday'] = daily.index.dayofweek  # Monday=0, Sunday=6
daily['is_weekend'] = daily['weekday'] >= 5

# ------------------------
# 2. Calculate average rides
# ------------------------
avg_weekday = daily[daily['is_weekend'] == False]['total'].mean()
avg_weekend = daily[daily['is_weekend'] == True]['total'].mean()

print('Average rides on weekdays:', round(avg_weekday, 0))
print('Average rides on weekends:', round(avg_weekend, 0))

# ------------------------
# 3. Bar chart
# ------------------------
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Weekday', 'Weekend'], [avg_weekday, avg_weekend], color=['steelblue', 'coral'])
ax.set_title('Average Daily Rides: Weekday vs Weekend (Jan–Dec 2025)')
ax.set_ylabel('Average Total Rides')
ax.set_xlabel('Day Type')
ax.text(0, avg_weekday + 1000, f'{avg_weekday:,.0f}', ha='center', fontsize=10)
ax.text(1, avg_weekend + 1000, f'{avg_weekend:,.0f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()



## Step 7 · Average Rides by Hour of Day

Look at how ride volume changes throughout the day.

In [ ]:
# ------------------------
# 1. Calculate average rides per hour
# ------------------------
hourly_avg = hourly['total'].groupby(level='hour').mean()

# ------------------------
# 2. Line plot
# ------------------------
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly_avg.index, hourly_avg.values, color='steelblue', marker='o', linewidth=2)
ax.set_title('Average Rides by Hour of Day (Jan–Dec 2025)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Rides')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()


## Step 8 · Peak Hours Identification

Identify the top morning and evening commute peak hours based on average ride volume.

In [ ]:
# ------------------------
# 1. Find top 3 morning peak hours (6am - 11am)
# ------------------------
morning = hourly_avg.loc[6:11]
morning_peak = morning.nlargest(3)
print('Top morning peak hours:')
print(morning_peak)

# ------------------------
# 2. Find top 3 evening peak hours (4pm - 9pm)
# ------------------------
evening = hourly_avg.loc[16:21]
evening_peak = evening.nlargest(3)
print('\nTop evening peak hours:')
print(evening_peak)

# ------------------------
# 3. Bar chart of all hours, highlight peaks
# ------------------------
colors = ['steelblue'] * 24
for h in morning_peak.index:
    colors[h] = 'orange'
for h in evening_peak.index:
    colors[h] = 'coral'

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(hourly_avg.index, hourly_avg.values, color=colors)
ax.set_title('Average Rides by Hour — Peak Hours Highlighted (Jan–Dec 2025)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Rides')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()


## Step 9 · Weekday-by-Hour Heatmap

Show the average ride volume for each combination of day of week and hour of day.

In [ ]:
# ------------------------
# 1. Add day of week to hourly data
# ------------------------
hourly_reset = hourly.reset_index()
hourly_reset['weekday'] = hourly_reset['date'].dt.dayofweek

# ------------------------
# 2. Calculate average rides by weekday and hour
# ------------------------
heatmap_data = hourly_reset.groupby(['weekday', 'hour'])['total'].mean().unstack()

# Rename rows to day names
heatmap_data.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# ------------------------
# 3. Plot heatmap
# ------------------------
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Average Rides by Day of Week and Hour (Jan–Dec 2025)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Day of Week')
plt.tight_layout()
plt.show()


## Step 10 · Member vs Casual Usage Patterns

Compare how member and casual users ride differently by hour of day and by day of week. The first plot uses average hourly volume; the second uses average daily volume.


In [ ]:
# ------------------------
# 1. Average rides by hour for each user type
# ------------------------
member_by_hour = hourly['member'].groupby(level='hour').mean()
casual_by_hour = hourly['casual'].groupby(level='hour').mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(member_by_hour.index, member_by_hour.values, color='steelblue', marker='o', label='Member')
ax.plot(casual_by_hour.index, casual_by_hour.values, color='coral', marker='o', label='Casual')
ax.set_title('Average Rides by Hour — Member vs Casual (Jan–Dec 2025)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Rides')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

# ------------------------
# 2. Average daily rides by day of week for each user type
# ------------------------
daily_by_user = daily.copy()
daily_by_user['weekday'] = daily_by_user.index.dayofweek

member_by_day = daily_by_user.groupby('weekday')['member'].mean()
casual_by_day = daily_by_user.groupby('weekday')['casual'].mean()

day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(day_labels, member_by_day.values, color='steelblue', marker='o', label='Member')
ax.plot(day_labels, casual_by_day.values, color='coral', marker='o', label='Casual')
ax.set_title('Average Daily Rides by Day of Week — Member vs Casual (Jan–Dec 2025)')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Average Daily Rides')
ax.legend()
plt.tight_layout()
plt.show()


# 5. Addressing the Question

## Step 11 · Create Forecasting Dataset

Use the cleaned daily ride counts as the forecasting dataset.

In [ ]:
# ------------------------
# 1. Use daily total rides as the forecast target
# ------------------------
forecast_df = daily[['total']].copy()

print('Forecasting dataset shape:', forecast_df.shape)
print('Date range:', forecast_df.index.min().date(), 'to', forecast_df.index.max().date())
print()
print(forecast_df.head())


## Step 12 · Add Time-Based Features

Add calendar features, lag features, and rolling averages to the forecasting dataset.

In [ ]:
# ------------------------
# 1. Calendar features
# ------------------------
forecast_df['dayofweek'] = forecast_df.index.dayofweek
forecast_df['month']     = forecast_df.index.month
forecast_df['is_weekend'] = (forecast_df['dayofweek'] >= 5).astype(int)

# ------------------------
# 2. Lag features
# ------------------------
forecast_df['lag_1']  = forecast_df['total'].shift(1)
forecast_df['lag_7']  = forecast_df['total'].shift(7)

# ------------------------
# 3. Rolling average (7-day)
# ------------------------
forecast_df['rolling_7'] = forecast_df['total'].shift(1).rolling(window=7).mean()

# ------------------------
# 4. Drop rows with NaN (from lag/rolling)
# ------------------------
forecast_df = forecast_df.dropna()

print('Shape after adding features:', forecast_df.shape)
print(forecast_df.head())


## Step 13 - Train / Test Splits

Use two separate monthly holdout tests:
- **June 2025**: train on all available data before June 1, then test June 1-30.
- **December 2025**: train on all available data before December 1, then test December 1-31.

This checks whether the models perform well in both a high-demand summer month and a lower-demand winter month.

In [ ]:
# ------------------------
# 1. Define reusable monthly train/test splits
# ------------------------
TEST_MONTHS = ["2025-06", "2025-12"]
features = ["dayofweek", "month", "is_weekend", "lag_1", "lag_7", "rolling_7"]


def get_train_test(df, test_month):
    test_start = pd.Timestamp(f"{test_month}-01")
    test_end = test_start + pd.offsets.MonthEnd(0)

    train_df = df[df.index < test_start]
    test_df = df[(df.index >= test_start) & (df.index <= test_end)]

    return train_df, test_df


splits = {}
for test_month in TEST_MONTHS:
    train_df, test_df = get_train_test(forecast_df, test_month)
    splits[test_month] = {"train": train_df, "test": test_df}

    print(f"Test month: {test_month}")
    print("Train:", train_df.shape, "|", train_df.index.min().date(), "to", train_df.index.max().date())
    print("Test: ", test_df.shape, "|", test_df.index.min().date(), "to", test_df.index.max().date())
    print()

# Clear named aliases so the two holdout sets are easy to inspect later.
june_train = splits["2025-06"]["train"]
june_test = splits["2025-06"]["test"]
december_train = splits["2025-12"]["train"]
december_test = splits["2025-12"]["test"]

split_summary = pd.DataFrame([
    {
        "Test Month": month,
        "Train Start": split["train"].index.min().date(),
        "Train End": split["train"].index.max().date(),
        "Train Days": len(split["train"]),
        "Test Start": split["test"].index.min().date(),
        "Test End": split["test"].index.max().date(),
        "Test Days": len(split["test"]),
    }
    for month, split in splits.items()
])

split_summary

## Step 14 - Baseline Models

Three simple baselines are generated separately for each test month:
- **Moving Average**: predict using the mean of the last 7 training days
- **Previous Day**: predict today = yesterday's actual
- **Previous Week Same Day**: predict today = same weekday one week ago

In [ ]:
# ------------------------
# 1. Baseline predictions for each test month
# ------------------------
def make_baseline_predictions(df, train_df, test_df):
    ma_value = train_df["total"].iloc[-7:].mean()
    pred_ma = np.repeat(ma_value, len(test_df))

    prev_day = df["total"].shift(1)
    pred_prev_day = prev_day.loc[test_df.index].values

    prev_week = df["total"].shift(7)
    pred_prev_week = prev_week.loc[test_df.index].values

    return {
        "Moving Average": pred_ma,
        "Previous Day": pred_prev_day,
        "Previous Week Same Day": pred_prev_week,
    }


baseline_predictions = {}
for test_month, split in splits.items():
    baseline_predictions[test_month] = make_baseline_predictions(
        forecast_df,
        split["train"],
        split["test"],
    )

    print(f"{test_month} moving average prediction:", round(baseline_predictions[test_month]["Moving Average"][0], 1))
    print(f"{test_month} previous-day sample:", baseline_predictions[test_month]["Previous Day"][:5].round(0))
    print()

## Step 15 - Linear Regression Model

Train a separate linear regression for each test month using only the observations available before that month, then predict the corresponding monthly holdout set.

In [ ]:
# ------------------------
# 1. Fit and predict linear regression for each test month
# ------------------------
lr_predictions = {}

for test_month, split in splits.items():
    train_df = split["train"]
    test_df = split["test"]

    X_train = train_df[features]
    y_train = train_df["total"]
    X_test = test_df[features]

    lr = LinearRegression()
    lr.fit(X_train, y_train)
    lr_predictions[test_month] = lr.predict(X_test)

    print(f"Linear regression trained for {test_month}.")
    print("Sample predictions:", lr_predictions[test_month][:5].round(0))
    print()

## Step 15B - ACF/PACF Diagnostics for ARIMA

Use the January-November training period for ARIMA diagnostics. First differencing removes the strongest trend, and the ACF/PACF plots motivate a simple ARIMA(1,1,1) specification for the rolling forecasts.

In [ ]:
# ------------------------
# 1. Difference the January-November training series for ARIMA diagnostics
# ------------------------
diagnostic_train = forecast_df[forecast_df.index < "2025-12-01"]["total"].asfreq("D")
arima_diff = diagnostic_train.diff().dropna()

# ------------------------
# 2. Plot ACF and PACF
# ------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(arima_diff, lags=30, ax=axes[0])
plot_pacf(arima_diff, lags=30, ax=axes[1], method="ywm")
axes[0].set_title("ACF of Differenced Training Series")
axes[1].set_title("PACF of Differenced Training Series")
plt.tight_layout()
plt.show()

print("ARIMA order used for forecasting: (1, 1, 1)")
print("d = 1 is used because the daily series has a clear trend over the training period.")
print("p = 1 and q = 1 provide a simple specification consistent with short-lag dependence in the diagnostics.")

## Step 16 - ARIMA Model

Fit a rolling ARIMA(1,1,1) model for each test month. For every test date, the model is trained on observations available up to the previous day, forecasts one day ahead, and then updates the history with the observed value.

In [ ]:
# ------------------------
# 1. Rolling one-step-ahead ARIMA forecast for each test month
# ------------------------
def make_arima_predictions(train_df, test_df):
    history = train_df["total"].asfreq("D").copy()
    predictions = []

    for current_date in test_df.index:
        arima_model = ARIMA(history.asfreq("D"), order=(1, 1, 1))
        arima_result = arima_model.fit()
        next_forecast = arima_result.forecast(steps=1).iloc[0]
        predictions.append(next_forecast)

        # After forecasting this date, reveal the observed value for the next step.
        history.loc[current_date] = test_df.loc[current_date, "total"]

    return np.array(predictions)


arima_predictions = {}
for test_month, split in splits.items():
    arima_predictions[test_month] = make_arima_predictions(split["train"], split["test"])

    print(f"Rolling ARIMA(1,1,1) fitted for {test_month}.")
    print("Sample forecast:", arima_predictions[test_month][:5].round(0))
    print()

## Step 17 - Model Evaluation

Compare all models using MAE, MSE, and RMSE on both June 2025 and December 2025. The final summary reports model performance by test month and the average performance across the two holdout months.

In [ ]:
# ------------------------
# 1. Compute metrics for each model and each test month
# ------------------------
def eval_model(test_month, name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    return {
        "Test Month": test_month,
        "Model": name,
        "MAE": round(mae, 1),
        "MSE": round(mse, 1),
        "RMSE": round(rmse, 1),
    }


all_predictions = {}
results = []

for test_month, split in splits.items():
    test_df = split["test"]
    y_true = test_df["total"].values

    model_predictions = {
        **baseline_predictions[test_month],
        "Linear Regression": lr_predictions[test_month],
        "Rolling ARIMA(1,1,1)": arima_predictions[test_month],
    }
    all_predictions[test_month] = model_predictions

    for name, pred in model_predictions.items():
        results.append(eval_model(test_month, name, y_true, pred))

results_df = pd.DataFrame(results)
results_by_month = results_df.sort_values(["Test Month", "MAE"]).set_index(["Test Month", "Model"])
summary_df = results_df.groupby("Model")[["MAE", "MSE", "RMSE"]].mean().sort_values("MAE").round(1)

print("Results by test month:")
print(results_by_month.to_string())
print()
print("Average across June and December test months:")
print(summary_df.to_string())

# Show the table as the final object in the output area for easier notebook inspection.
results_by_month

In [ ]:
# ------------------------
# 2. Plot actual vs predicted for each test month
# ------------------------
for test_month, split in splits.items():
    test_df = split["test"]
    y_true = test_df["total"].values
    model_predictions = all_predictions[test_month]

    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    axes = axes.flatten()

    for i, (name, pred) in enumerate(model_predictions.items()):
        axes[i].plot(test_df.index, y_true, label="Actual", color="black", linewidth=2)
        axes[i].plot(test_df.index, pred, label="Predicted", color="steelblue", linestyle="--")
        axes[i].set_title(name)
        axes[i].set_xlabel("Date")
        axes[i].set_ylabel("Total Rides")
        axes[i].legend()

    axes[5].set_visible(False)

    plt.suptitle(f"Actual vs Predicted Daily Rides - {test_month}", fontsize=14)
    plt.tight_layout()
    plt.show()

# ------------------------
# 3. Residual sanity check for the best model in each test month
# ------------------------
for test_month, split in splits.items():
    test_df = split["test"]
    monthly_results = results_df[results_df["Test Month"] == test_month].sort_values("MAE")
    best_model_name = monthly_results.iloc[0]["Model"]
    best_pred = all_predictions[test_month][best_model_name]

    residuals = pd.Series(test_df["total"].values - best_pred, index=test_df.index, name="residual")
    residual_mean = residuals.mean()
    residual_std = residuals.std()
    outside_3sigma = residuals[np.abs(residuals - residual_mean) > 3 * residual_std]

    print(f"Best model by MAE for {test_month}: {best_model_name}")
    print("Residual mean:", round(residual_mean, 1))
    print("Residual std :", round(residual_std, 1))
    print("Residuals outside +/-3 sigma:", len(outside_3sigma))
    if len(outside_3sigma) > 0:
        print("Dates outside +/-3 sigma:", list(outside_3sigma.index.date))
    print()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(residuals.index, residuals, marker="o", color="darkorange")
    ax.axhline(residual_mean, color="black", linewidth=1, label="Mean residual")
    ax.axhline(residual_mean + 3 * residual_std, color="red", linestyle="--", label="+/-3 sigma")
    ax.axhline(residual_mean - 3 * residual_std, color="red", linestyle="--")
    ax.set_title(f"Residual Check - {best_model_name} ({test_month})")
    ax.set_xlabel("Date")
    ax.set_ylabel("Actual - Predicted")
    ax.legend()
    plt.tight_layout()
    plt.show()